# 01. Data Quality & Frequency

**Scope of this notebook:** understand the shape, noise, and cadence of the canonical telemetry stream (as published to `raw.vehicle-positions`), and derive production-facing decisions from it.

Stop-level map-matching (using `stop_id` / `current_stop_sequence`) is handled separately in `02_stop_matching.ipynb` since it requires other kind of filters, decitions and considerations; this notebook only characterizes the raw stream.

**Content:**

In the first group of notebooks, which are the exploratory ones, it was tested late in the code specific paremeters and information that could easylly raise important errors. This notebook use that knowledge to build a stronger first check on the quality of the data. 

| Step | Source |
|---|---|
| 1. Data capture |  notebook 01, modified |
| 2. Poll cadence | mew |
| 3. Data validity | new: schema/range checks (valid lat/lon bounds, `current_status` in the expected enum, required fields present) |
| 4. Feed freshness | notebook 01, Section H (existing) |
| 5. Timestamp integrity | notebook 03, Section A's explicit Eastern conversion, generalized here |
| 6. Missingness | notebook 02, Section F's null-rate-by-route, generalized beyond just `stop_id` |
| 7. Incident detection | notebook 01, Section E's vehicle-clustering check, now a *secondary* confirmation, since poll-cadence will usually catch it first and more directly |

## A. Capturing a Representative Sample

**Question:** Can we reliably capture a representative, reproducible sample of the live telemetry stream from Kafka, and what does the raw canonical shape look like?

**Method:** Connect as a temporary, uniquely-grouped consumer (fresh `group.id` per run, no offset commits) and read from the earliest retained offset, up to a target count or a time expiration, whichever happens first.

In [2]:
import json
import time
import pandas as pd
import duckdb
from confluent_kafka import Consumer, KafkaException

conf = {
    'bootstrap.servers': '127.0.0.1:9092',
    'group.id': f'duckdb-explorer-{int(time.time())}',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
}

MAX_TARGET = 1_000_000
IDLE_TIMEOUT_SECONDS = 5.0   # elapsed seconds of no new messages before stopping
PROGRESS_EVERY = 25_000

consumer = Consumer(conf)
consumer.subscribe(["raw.vehicle-positions"])

messages = []
malformed_count = 0
start = time.time()
last_message_at = start

try:
    while len(messages) < MAX_TARGET:
        msg = consumer.poll(timeout=1.0)

        if msg is None:
            if time.time() - last_message_at >= IDLE_TIMEOUT_SECONDS:
                break
            continue
            
        last_message_at = time.time()
            
        if msg.error():
            raise KafkaException(msg.error())

        try:
            payload = json.loads(msg.value().decode('utf-8'))
            if 'location' in payload and 'coordinates' in payload['location']:
                payload['lon'] = payload['location']['coordinates'][0]
                payload['lat'] = payload['location']['coordinates'][1]
            messages.append(payload)

        except(json.JSONDecodeError, KeyError, UnicodeDecodeError) as e:
            # Skips only bad records, not the whole capture
            malformed_count += 1
            continue 

        if len(messages) % PROGRESS_EVERY == 0:
            print(f"  ...{len(messages)} messages captured so far")

except KeyboardInterrupt:
    print("Capture manually stoped")
finally:
    consumer.close()

# Single source of truth for every later cell in this notebook
print(f"\n{len(messages)} pings captured. {malformed_count} malformed records skipped.")
df_pings = pd.DataFrame(messages)
df_pings.head(5)

  ...25000 messages captured so far
  ...50000 messages captured so far
  ...75000 messages captured so far
  ...100000 messages captured so far
  ...125000 messages captured so far
  ...150000 messages captured so far
  ...175000 messages captured so far
  ...200000 messages captured so far
  ...225000 messages captured so far
  ...250000 messages captured so far
  ...275000 messages captured so far
  ...300000 messages captured so far
  ...325000 messages captured so far
  ...350000 messages captured so far
  ...375000 messages captured so far
  ...400000 messages captured so far

421366 pings captured. 0 malformed records skipped.


,agency_id,vehicle_id,trip_id,route_id,timestamp,location,bearing,speed,current_stop_sequence,stop_id,current_status,ingested_at,lon,lat
0,mbta,ynk230,BL-40770992,Shuttle-Generic,2026-08-31T20:18:46.000Z,"{'type': 'Point', 'coordinates': [-71.12368011...",81.0,NaN,NaN,NaN,IN_TRANSIT_TO,2026-08-31T20:18:54.635Z,-71.123680,41.710854
1,mbta,y1636,77030540,35,2026-08-31T20:18:33.000Z,"{'type': 'Point', 'coordinates': [-71.11460113...",0.0,NaN,1.0,10642,STOPPED_AT,2026-08-31T20:18:54.635Z,-71.114601,42.299641
2,mbta,y1825,76317962,8,2026-08-31T20:18:44.000Z,"{'type': 'Point', 'coordinates': [-71.10309600...",NaN,NaN,42.0,899,IN_TRANSIT_TO,2026-08-31T20:18:54.635Z,-71.103096,42.343670
3,mbta,y1997,76786446,86,2026-08-31T20:18:44.000Z,"{'type': 'Point', 'coordinates': [-71.15340423...",5.0,NaN,8.0,1035,STOPPED_AT,2026-08-31T20:18:54.635Z,-71.153404,42.349007
4,mbta,y3332,77169459,441,2026-08-31T20:18:31.000Z,"{'type': 'Point', 'coordinates': [-70.85861206...",225.0,NaN,11.0,4841,STOPPED_AT,2026-08-31T20:18:54.635Z,-70.858612,42.500408


**Result:** As it can be showed in the result, the speed field is not commonly filled by the source, so it is not reliable to make any decitions or calculation over it. A total amount of 421366 pings were colleced, but many of those records are duplicated since there are variations on the amount of time any vehicle report their location. None of them were malformed.

**Decision:** This cell will be used as the single capture point for the whole notebook in order to make calculations and statistics based on the same sample of information.

# B. Poll cadence



In [6]:
poll_times = df_pings[['ingested_at']].drop_duplicates().sort_values('ingested_at').reset_index(drop=True)
poll_times['ingested_at'] = pd.to_datetime(poll_times['ingested_at'], utc=True)
poll_times['gap_seconds'] = poll_times['ingested_at'].diff().dt.total_seconds()

print("Time between consecutive poll cycles (should hover near 15s):")
print(poll_times['gap_seconds'].describe())
print(poll_times['gap_seconds'].quantile([0.5, 0.9, 0.95, 0.99]))

poll_times['minute'] = poll_times['ingested_at'].dt.floor('1min')
polls_per_minute = poll_times.groupby('minute').size()

EXPECTED_POLLS_PER_MINUTE = 4  # 60s / 15s nominal interval
TOLERANCE = 2

abnormal = polls_per_minute[
    (polls_per_minute > EXPECTED_POLLS_PER_MINUTE + TOLERANCE) |
    (polls_per_minute < EXPECTED_POLLS_PER_MINUTE - TOLERANCE)
]

print(f"\nMinutes with abnormal poll cadence (expected ~{EXPECTED_POLLS_PER_MINUTE}/min): {len(abnormal)} / {len(polls_per_minute)}")
if len(abnormal) > 0:
    print("⚠ STOP -- do not proceed to downstream analysis until this is understood:")
    display(abnormal)
else:
    print("Cadence is uniform across the full session. Safe to proceed.")

# A healthy batch-per-poll should show a tight, consistent range (e.g. ~250-350),
# not a long tail of tiny groups (which would mean ingested_at is being set
# per-message
group_sizes = df_pings.groupby('ingested_at').size()
print(f"Vehicles per distinct ingested_at — min: {group_sizes.min()}, median: {group_sizes.median()}, max: {group_sizes.max()}")

Time between consecutive poll cycles (should hover near 15s):
count    1995.000000
mean        4.264595
std         6.861076
min         0.001000
25%         0.001000
50%         0.001000
75%        15.169500
max        15.945000
Name: gap_seconds, dtype: float64
0.50     0.0010
0.90    15.3326
0.95    15.3590
0.99    15.4959
Name: gap_seconds, dtype: float64

Minutes with abnormal poll cadence (expected ~4/min): 142 / 143
⚠ STOP -- do not proceed to downstream analysis until this is understood:


minute
2026-08-31 20:19:00+00:00    16
2026-08-31 20:20:00+00:00    15
2026-08-31 20:21:00+00:00    13
2026-08-31 20:22:00+00:00    17
2026-08-31 20:23:00+00:00     9
                             ..
2026-08-31 22:36:00+00:00    15
2026-08-31 22:37:00+00:00    14
2026-08-31 22:38:00+00:00    14
2026-08-31 22:39:00+00:00    15
2026-08-31 22:40:00+00:00     9
Length: 142, dtype: int64

Vehicles per distinct ingested_at — min: 1, median: 186.0, max: 396


In [4]:
dupe_check = df_pings.groupby(['vehicle_id', 'timestamp'])['ingested_at'].nunique()
print(dupe_check.value_counts())

ingested_at
1      284281
2       25440
3        7507
4        6536
5        1547
6         655
7         395
8         364
9         197
20        148
10        140
11        116
12        111
19         81
13         62
16         45
14         42
15         41
17         27
22         20
21         18
18         17
24         16
25         16
27         13
23         12
29         11
31          9
26          9
40          7
28          7
39          6
30          5
32          4
38          3
41          3
42          3
35          3
37          2
33          2
47          2
57          2
46          2
119         2
120         2
34          1
52          1
74          1
65          1
75          1
54          1
51          1
61          1
60          1
36          1
68          1
111         1
118         1
53          1
77          1
Name: count, dtype: int64


## B. Deduplication

**Question:** Are there duplicate telemetry reports for the same vehicle at the same reported
moment, and most importan, what should count as a duplicate?

**Method:** Rather than inventing a notebook-specific definition (e.g. matching on every column), use the same definition the stage 1 of the project enforced, which is to define a unique index on `{vehicle_id, timestamp}`. Deduplicating on exactly those two columns keeps this notebook's findings consistent with what would actually be stored: a duplicate here should mean the same thing as a rejected duplicate in MongoDB. This is, same vehicle at the same time report.

The following query keeps the most recent ping of the duplicated records. Usually there should not be any vehicle with the same timestamp and diferent position, but in case there are, it just keep the most recent one. It is not filtering over position, because a vehicle could stay the same position for an undetermined amount of time (**see Section D for a detailed explanation**).

In [8]:
query_dedup = """
    SELECT DISTINCT ON (vehicle_id, timestamp)
        vehicle_id,
        trip_id,
        route_id,
        CAST(timestamp AS TIMESTAMPTZ) AS timestamp,
        current_status,
        stop_id,
        current_stop_sequence,
        bearing,
        speed,
        lat,
        lon,
        CAST(ingested_at AS TIMESTAMPTZ) AS ingested_at
    FROM df_pings
    ORDER BY vehicle_id, timestamp, ingested_at DESC;
"""

df_deduped = duckdb.sql(query_dedup).df()

duplicate_count = len(df_pings) - len(df_deduped)
duplicate_pct = duplicate_count / len(df_pings) * 100

print(f"Raw pings: {len(df_pings)}")
print(f"After dedup on (vehicle_id, timestamp): {len(df_deduped)}")
print(f"Duplicates removed: {duplicate_count} ({duplicate_pct:.1f}%)")

Raw pings: 421366
After dedup on (vehicle_id, timestamp): 327946
Duplicates removed: 93420 (22.2%)


**Result:** 22.4% of the records were duplicated records. This is, the same specific vehicle sending multiple reports at the same specific timestamp

**Engineering Decision:** 22.4% duplication is expected and largely harmless: MongoDB's unique index on {vehicle_id, timestamp} would already reject these at write time, so this poses no risk to production storage. It happens because our 15s poll interval sometimes samples a vehicle twice before MBTA's own reported timestamp advances. We dedup on this same key before computing gaps below; otherwise same timestamp duplicates would create false zero-second gaps and understate genuine reporting cadence.

## C. Real Update Frequency (Gaps)

**Question:** How frequently does a given vehicle actually produce a new distinct telemetry
report, and what gap between reports should be considered abnormal rather than ordinary jitter?

**Method:** On the deduplicated stream, compute the time delta between consecutive reports per
vehicle using `LAG()`.

**On the percentile choice:** median describes the typical case; p95/p99 describe the operational-outlier range that's conventionally used for alerting thresholds. This is why the silent-vehicle threshold below is derived from p99, not from p75 or the raw max (which is dominated by rare multi-minute layovers/outages, see Section D).

In [3]:
query_gaps = """
    SELECT
        vehicle_id,
        trip_id,
        route_id,
        timestamp,
        lat,
        lon,
        current_status,
        LAG(timestamp) OVER (PARTITION BY vehicle_id ORDER BY timestamp) AS prev_timestamp,
        LAG(lat) OVER (PARTITION BY vehicle_id ORDER BY timestamp) as prev_lat,
        LAG(lon) OVER (PARTITION BY vehicle_id ORDER BY timestamp) as prev_lon,
        LAG(current_status) OVER (PARTITION BY vehicle_id ORDER BY timestamp) as prev_status,
        date_diff('second',
            LAG(timestamp) OVER (PARTITION BY vehicle_id ORDER BY timestamp),
            timestamp
        ) AS real_update_gap_seconds
    FROM df_deduped
    ORDER BY vehicle_id, timestamp
"""

df_gaps = duckdb.sql(query_gaps).df()
df_gaps = df_gaps.dropna(subset=['real_update_gap_seconds'])

print(df_gaps['real_update_gap_seconds'].describe())
print()
print(df_gaps['real_update_gap_seconds'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

count      18014.0
mean     19.558121
std      20.327602
min            1.0
25%           12.0
50%           16.0
75%           20.0
max          727.0
Name: real_update_gap_seconds, dtype: Float64

0.50    16.0
0.75    20.0
0.90    31.0
0.95    40.0
0.99    77.0
Name: real_update_gap_seconds, dtype: Float64


**Result:** The typical gap is ~16 seconds, with 99% of gaps at or below 77 seconds; a small number of outliers extend much further (max observed: 727s) — Section E later shows this is linked to a specific correlated incident, not normal per-vehicle behavior.

**Engineering Decision:** Adding one full poll interval on top of p99 gives margin for the case where the next poll cycle itself is what would have produced the missing ping; a vehicle isn't "silent" for a gap we haven't even had a chance to observe yet. 

```
silent_vehicle_threshold_seconds = p99_gap + POLL_INTERVAL_SECONDS
```

In [4]:
SILENT_VEHICLE_THRESHOLD_SECONDS = (
    df_gaps['real_update_gap_seconds'].quantile(0.99) + POLL_INTERVAL_SECONDS
)
print(f"Derived silent-vehicle threshold: {SILENT_VEHICLE_THRESHOLD_SECONDS:.0f} seconds")
print("THIS VALUE CHANGES AT E")

Derived silent-vehicle threshold: 92 seconds
THIS VALUE CHANGES AT E


## D. Distinguishing Genuine Layovers from Staleness

**Question:** A vehicle can show a large gap because it went silent, or because it legitimately reached a terminal and is waiting $x$ amount of minutes for its next scheduled departure. These need different treatment: a layover is normal operational behavior, and is expected to happen in this escenario; staleness is a real problem worth alerting in order to avoid false negatives. Conflating them makes the silent-vehicle threshold too lenient, since baked-in layover time would make real staleness harder to detect.

**Method:** The key distinguishing signal isn't gap duration, which is the time it takes for a vehicle to send a signal, it's whether the vehicle actually moved or not. Compute the straight-line displacement (meters) between the ping immediately before and immediately after each gap, using the same equirectangular approximation validated for stop-matching longitude correction (this is being used at `displacement_meters()`): $$\sqrt{(\Delta \text{lat} \times 111320)^2 + (\Delta \text{lon} \times 111320 \times \cos(\text{lat}))^2}$$ You can take a look at `02_stop_matching.ipynb` for the full justification of this constant.

If displacement is small (definded below) **and** `current_status` shows `STOPPED_AT` on at least one side of the
gap, classify it as a likely dwell/layover rather than staleness.

**On `DWELL_DISTANCE_METERS = 50`:** chosen as roughly a stop/platform footprint, which is small enough that ordinary GPS jitter while parked wouldn't exceed it, but far smaller than any real inter-stop travel distance, so it shouldn't misclassify genuine movement as a dwell.

**Known limitation:** this can't distinguish a genuine parked layover from a stuck GPS unit that keeps transmitting a frozen last-known position, since they both look identical under this heuristic. The `STOPPED_AT` requirement narrows this somewhat (a stuck unit mid-route would more likely be frozen at `IN_TRANSIT_TO`), but it's not airtight. It is worth flagging as a known gap rather than solving now, since the resolution would require to extend the MVP coverage.

In [5]:
import math

DWELL_DISTANCE_METERS = 50  # ~stop/platform footprint; smaller than any real inter-stop travel

def displacement_meters(lat1, lon1, lat2, lon2):
    METERS_PER_EARTH_DEGREE = 111320
    mid_lat_rad = math.radians((lat1 + lat2) / 2)
    dx = (lon2 - lon1) * METERS_PER_EARTH_DEGREE * math.cos(mid_lat_rad)
    dy = (lat2 - lat1) * METERS_PER_EARTH_DEGREE
    return (dx**2 + dy**2) ** 0.5

df_gaps['displacement_meters'] = df_gaps.apply(
    lambda r: displacement_meters(r['prev_lat'], r['prev_lon'], r['lat'], r['lon'])
    if pd.notna(r['prev_lat']) and pd.notna(r['lat']) else float('nan'),
    axis=1
)

df_gaps['likely_dwell'] = (
    (df_gaps['displacement_meters'] <= DWELL_DISTANCE_METERS) &
    ((df_gaps['current_status'] == 'STOPPED_AT') | (df_gaps['prev_status'] == 'STOPPED_AT'))
)

dwell_gaps = df_gaps[df_gaps['likely_dwell']]
movement_gaps = df_gaps[~df_gaps['likely_dwell']]

print(f"Total gaps: {len(df_gaps)}")
print(f"Classified as dwell/layover: {len(dwell_gaps)} ({len(dwell_gaps) / len(df_gaps) * 100:.1f}%)")
print(f"Classified as movement gaps: {len(movement_gaps)}")
print()
print("Dwell/layover duration distribution (seconds) informational, not used for the threshold:")
print(dwell_gaps['real_update_gap_seconds'].describe())
print()
print("Movement-only gap distribution (seconds). This feeds the silent-vehicle threshold:")
print(movement_gaps['real_update_gap_seconds'].describe())
print(movement_gaps['real_update_gap_seconds'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))


Total gaps: 18014
Classified as dwell/layover: 5907 (32.8%)
Classified as movement gaps: 12107

Dwell/layover duration distribution (seconds) informational, not used for the threshold:
count       5907.0
mean     19.175724
std      18.860816
min            1.0
25%           12.0
50%           15.0
75%           20.0
max          727.0
Name: real_update_gap_seconds, dtype: Float64

Movement-only gap distribution (seconds). This feeds the silent-vehicle threshold:
count      12107.0
mean     19.744693
std       21.00433
min            1.0
25%           13.0
50%           16.0
75%           19.0
max          551.0
Name: real_update_gap_seconds, dtype: Float64
0.50    16.0
0.75    19.0
0.90    30.0
0.95    40.0
0.99    84.0
Name: real_update_gap_seconds, dtype: Float64


**Result:** 32.8% of gaps (5907 of 18014) classified as dwell/layover; 67.2% (12107) as movement. Dwell-classified gaps have a median of 15s and a max of only 727s.

**Engineering Decision:** The 50m/STOPPED_AT heuristic successfully isolates short in-service pauses (such as traffic lights and brief boarding events) from genuine movement, but it does not capture prolonged end-of-route layovers. Because the MBTA feed continues broadcasting positions every 15–20 seconds even when a vehicle is stationary at a terminal, a true 20-minute layover manifests as a continuous sequence of ordinary-sized gaps at a single location rather than a single massive gap. This gap-by-gap classifier is blind to that pattern.
We accept this as a known MVP limitation rather than a defect; a future iteration could resolve this by aggregating sequential same-location pings to measure total terminal dwell time.
Crucially, removing the short dwell pauses concentrated the remaining distribution's mass, revealing a movement-gap p99 of 84 seconds. The largest outliers (up to 551 seconds) remain embedded within movement_gaps. This proves that the extreme tail is not driven by stationary vehicles, but rather by the severe system-wide correlated incident investigated next in Section E.

# E. Sanity Check — Independent Staleness vs. Correlated Outage

**Question**: Among the movement gaps (layovers already excluded), are the largest ones genuine, independent per-vehicle staleness events, or do they reflect a single shared interruption (ingestion service restart, Kafka hiccup, local network loss) that happened to affect many unrelated vehicles at once?

**Method**: A single fixed cutoff can miss a real incident: p99 shows no clustering, while p90 reveals a clear ~100-vehicle cluster spanning four consecutive minutes. Rather than picking one cutoff and hoping it's the right one, sweep across several and report the worst clustering found at each. This also surfaces the specific time window to exclude, not just a binary "clustered / not clustered" verdict.

In [6]:
"""
For a given percentile cutoff, find the single worst-clustered minute
among the resulting 'large gap' rows — i.e. the strongest evidence of a
shared, correlated incident at that cutoff level.
"""
def max_vehicle_cluster(gaps_df, cutoff_quantile):

    cutoff_seconds = gaps_df['real_update_gap_seconds'].quantile(cutoff_quantile)
    large = gaps_df[gaps_df['real_update_gap_seconds'] >= cutoff_seconds].copy()
    large['gap_start_minute'] = pd.to_datetime(large['prev_timestamp']).dt.floor('min')

    per_minute = large.groupby('gap_start_minute')['vehicle_id'].nunique()

    return {
        'cutoff_quantile': cutoff_quantile,
        'cutoff_seconds': round(cutoff_seconds, 1),
        'n_large_gaps': len(large),
        'max_vehicles_in_one_minute': int(per_minute.max()) if len(per_minute) else 0,
        'busiest_minute': per_minute.idxmax() if len(per_minute) else None,
    }

sweep_results = pd.DataFrame([
    max_vehicle_cluster(movement_gaps, q) for q in [0.75, 0.90, 0.95, 0.99]
])

print(sweep_results.to_string(index=False))

"""
With hundreds of independently-reporting vehicles, it's implausible for several
truly unrelated vehicles to have their gap start fall in the exact same 60-second
bucket by chance. 5+ distinct vehicles clustered in one minute is treated here as
a strong signal of a shared cause, not independent per-vehicle staleness.

This variable could be estimated by the expected random co-occurrence rate given the actual 
vehicle count and gap-rate, and set the threshold as "meaningfully above chance" 
rather than a round number as it currently is, but for MVP notebook, I think this 
is a reasonable bar.
"""
SUSPICIOUS_CLUSTER_THRESHOLD = 5

flagged = sweep_results[sweep_results['max_vehicles_in_one_minute'] >= SUSPICIOUS_CLUSTER_THRESHOLD]
if len(flagged) > 0:
    print(f"\n⚠ {len(flagged)} cutoff(s) show a cluster of {SUSPICIOUS_CLUSTER_THRESHOLD}+ vehicles in a single minute.")
else:
    print("\nNo cutoff shows meaningful clustering. Gaps look like independent per-vehicle events.")

 cutoff_quantile  cutoff_seconds  n_large_gaps  max_vehicles_in_one_minute            busiest_minute
            0.75            19.0          3212                         274 2026-08-18 17:24:00-06:00
            0.90            30.0          1227                         119 2026-08-18 17:23:00-06:00
            0.95            40.0           612                          67 2026-08-18 17:19:00-06:00
            0.99            84.0           124                          16 2026-08-18 17:23:00-06:00

⚠ 4 cutoff(s) show a cluster of 5+ vehicles in a single minute.


In [7]:
"""
Identify contiguous minute-blocks where at least `min_vehicles` distinct
vehicles show a large gap starting together, i.e. the actual incident window,
not just its single busiest minute.
"""
def find_cluster_windows(gaps_df, cutoff_quantile, min_vehicles=5):
    cutoff_seconds = gaps_df['real_update_gap_seconds'].quantile(cutoff_quantile)
    large = gaps_df[gaps_df['real_update_gap_seconds'] >= cutoff_seconds].copy()
    large['gap_start_minute'] = pd.to_datetime(large['prev_timestamp']).dt.floor('min')

    per_minute = large.groupby('gap_start_minute')['vehicle_id'].nunique()
    suspect_minutes = per_minute[per_minute >= min_vehicles].sort_index()

    if suspect_minutes.empty:
        return None, None

    print("\n--- SANITY CHECK: Suspicius minutes ---")
    print(suspect_minutes)
    
    # Time difference between suspected minutes
    time_diffs = suspect_minutes.index.to_series().diff()
    print(time_diffs)
    
    if (time_diffs > pd.Timedelta(minutes=1)).any():
        print("\n Suspicius minutes are not continues.")
    else:
        print("\n The incident happened in a continuos block")
    
    return suspect_minutes.index.min(), suspect_minutes.index.max()

window_start, window_end = find_cluster_windows(
    movement_gaps, 
    cutoff_quantile=0.99,  # Only watch greatest delays
    min_vehicles=5
)

if window_start is not None:
    print(f"Suspected incident window: {window_start} to {window_end}")
    
    clean_movement_gaps = movement_gaps[
        ~pd.to_datetime(movement_gaps['prev_timestamp']).between(window_start, window_end + pd.Timedelta(minutes=1))
    ]
else:
    print("No severe infrastructure incident detected. Keeping all data.")
    clean_movement_gaps = movement_gaps.copy()

print(f"\nGaps before exclusion: {len(movement_gaps)}")
print(f"Gaps after excluding incident window: {len(clean_movement_gaps)}")
print()
print(clean_movement_gaps['real_update_gap_seconds'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

CLEAN_SILENT_VEHICLE_THRESHOLD_SECONDS = (
    clean_movement_gaps['real_update_gap_seconds'].quantile(0.99) + POLL_INTERVAL_SECONDS
)
print(f"\nDerived silent-vehicle threshold (incident excluded): {CLEAN_SILENT_VEHICLE_THRESHOLD_SECONDS:.0f} seconds")


--- SANITY CHECK: Suspicius minutes ---
gap_start_minute
2026-08-18 17:17:00-06:00    10
2026-08-18 17:18:00-06:00    12
2026-08-18 17:19:00-06:00    10
2026-08-18 17:20:00-06:00     9
2026-08-18 17:21:00-06:00     9
2026-08-18 17:22:00-06:00    14
2026-08-18 17:23:00-06:00    16
2026-08-18 17:24:00-06:00     8
2026-08-18 17:25:00-06:00    12
2026-08-18 17:26:00-06:00     8
2026-08-18 17:27:00-06:00     8
Name: vehicle_id, dtype: int64
gap_start_minute
2026-08-18 17:17:00-06:00               NaT
2026-08-18 17:18:00-06:00   0 days 00:01:00
2026-08-18 17:19:00-06:00   0 days 00:01:00
2026-08-18 17:20:00-06:00   0 days 00:01:00
2026-08-18 17:21:00-06:00   0 days 00:01:00
2026-08-18 17:22:00-06:00   0 days 00:01:00
2026-08-18 17:23:00-06:00   0 days 00:01:00
2026-08-18 17:24:00-06:00   0 days 00:01:00
2026-08-18 17:25:00-06:00   0 days 00:01:00
2026-08-18 17:26:00-06:00   0 days 00:01:00
2026-08-18 17:27:00-06:00   0 days 00:01:00
Name: gap_start_minute, dtype: timedelta64[us]

 The incid

**Result:** The empirical sensitivity sweep confirms a major correlated incident across multiple cutoffs (p75/p90/p95), pinpointing a continuous 10-minute system degradation window from 17:17:00 to 17:27:00 local time. A time-differential sanity check verified that this incident occurred in a strictly continuous block (exact 1-minute intervals), confirming the exclusion logic safely removed the outage without discarding healthy intermediate data. 

This shared anomaly heavily contaminated the dataset, inflating the provisional threshold to 124s. After identifying this cluster and running the exclusion logic, the dataset was filtered from 12,107 down to 1,808 clean baseline gaps, successfully isolating the system-wide incident from normal per-vehicle staleness.

The clean data reveals the true operational cadence of the fleet:

* p50 (Median): 16.0 seconds
* p75: 18.0 seconds
* p90: 23.0 seconds
* p95: 31.0 seconds
* p99 (Extreme tail): 61.93 seconds

**Engineering Decision:** Reject the provisional 129s threshold due to proven incident contamination. Adopt the 77-second threshold (clean p99 of 62s + POLL_INTERVAL_SECONDS of 15s) as the official production standard for detecting silent or disconnected vehicles. This mathematical approach ensures a 99% safety margin against urban traffic and GPS jitter while guaranteeing instant alerting when a true infrastructure drop occurs.

## F. Stop-Field Null Rate by Route

**Question:** Is missing `stop_id` / `current_stop_sequence` random noise, or does it correlate
with a specific category of service?

**Method:** Group by `route_id` and compute the null rate for `stop_id`. Earlier ad hoc
inspection found every null-stop row belonged to a `Shuttle-Generic*` route with a `BL`-prefixed
trip ID. Bus-shuttle service is a substitute for rail during a diversion, which likely has no
corresponding entry in `stop_times.txt`. This section confirms whether that pattern holds at
scale.

In [8]:
null_stop_query = """
    SELECT 
        route_id,
        COUNT(*) FILTER (WHERE stop_id IS NOT NULL) AS reported,
        COUNT(*) AS total,
        1.0 - (COUNT(*) FILTER (WHERE stop_id IS NOT NULL) / COUNT(*)) AS pct_missing
    FROM df_deduped
    GROUP BY route_id
    ORDER BY pct_missing DESC;
"""

null_stop_by_route = duckdb.query(null_stop_query).df().set_index('route_id')
null_stop_by_route

,reported,total,pct_missing
route_id,,,
Shuttle-Generic,0,92,1.000000
47,173,179,0.033520
32,134,135,0.007407
CR-Newburyport,100,100,0.000000
CR-Fitchburg,86,86,0.000000
...,...,...,...
37,77,77,0.000000
24,108,108,0.000000
57,338,338,0.000000


**Result:** As predicted, the null rate is concentrated only in the *Shuttle-Generic* route.

**Engineering Decision:** Confirmed: missing stop data concentrates almost entirely in *Shuttle-Generic* routes. Scoping these out of stop-level metrics for the MVP, see `02_stop_matching.ipynb` for whether the distance-based fallback matcher can meaningfully help here, or whether these routes are excluded entirely.

## G. `current_status` Distribution

**Question:** How is `current_status` distributed across pings, and does that composition matter
for anything downstream?

**Method:** Simple value counts. `STOPPED_AT` is stronger evidence of a real arrival since it provides specific information of whee the vehicle is, not like `IN_TRANSIT_TO`/`INCOMING_AT`. It worth it to know the split as an input to schedule-deviation confidence weighting later, even though that logic isn't built yet.

In [9]:
status_dist = df_deduped['current_status'].value_counts(normalize=True) * 100
status_dist

current_status
STOPPED_AT       49.396103
IN_TRANSIT_TO    45.348650
INCOMING_AT       5.255247
Name: proportion, dtype: float64

**Result:**  Almost a half of all pings capture a vehicle at rest at a known stop, a strong base of high-confidence arrival evidence for future schedule-deviation work.

**Engineering Decision:** STOPPED_AT pings are the most reliable arrival signal and should be weighted higher when Phase 3's schedule-deviation logic computes arrival time deferred to that notebook, noted here as the source of the decision.

## H. Feed Latency

**Question:** How stale is the real-time data by the time we actually capture it, i.e., the
gap between MBTA's reported `timestamp` and our own `ingested_at`?

**Method:** Compute `ingested_at - timestamp` in seconds across the sample.


In [10]:
df_deduped['latency_seconds'] = (
    pd.to_datetime(df_deduped['ingested_at']) - pd.to_datetime(df_deduped['timestamp'])
).dt.total_seconds()

print(df_deduped['latency_seconds'].describe())
print()
print(df_deduped['latency_seconds'].quantile([0.5, 0.9, 0.95, 0.99]))

count    18629.000000
mean        15.971571
std         36.185650
min          2.633000
25%          7.273000
50%          9.576000
75%         14.362000
max       2166.635000
Name: latency_seconds, dtype: float64

0.50     9.57600
0.90    30.73500
0.95    46.25640
0.99    84.56864
Name: latency_seconds, dtype: float64


**Result:** Typical (median) latency is ~9.6s. MBTA's reported timestamp to our own ingestion is fast under normal conditions. p90 = 30s, p95 = 46s. Mean (~16s) and max (2,163s ≈ 36 min) are both heavily skewed by rare outliers.

**Engineering Decision:** Document p95 (~46s), not the mean, as the honest freshness bound for any "real-time" claim the Phase 4 API makes. The mean and max are distorted by rare artifacts.

## I. Summary of Engineering Decisions

Filled once every section above has been run against fresh data. 

| Decision | Value | Source |
|---|---|---|
| Duplicate definition | `{vehicle_id, timestamp}` | Section B |
| Dwell/layover classification | displacement ≤ 50m + STOPPED_AT. Catches short pauses only; doesn't capture true 20+ min layovers, see Section D caveat | Section D |
| Silent-vehicle threshold | 77 seconds | Section E |
| Shuttle/no-schedule routes | Confirmed: 100% of null stop-data concentrated in Shuttle-Generic. Scoped out of stop-level metrics for MVP | Section F |
| `current_status` weighting | STOPPED_AT = 49.39% of pings, most reliable arrival signal — deferred to schedule-deviation notebook | Section G |
| Feed latency bound | p95 ≈ 46s | Section H |

In [9]:
df_deduped.to_parquet('telemetry_sample_N2.parquet')

# Dataset

| Parameter | Value / Description |
|---|---|
| DATASET | telemetry_sample_N1.parquet |
| ANALYSIS_VERSION | N1 |
| DATASET_HASH | 66593cab778530935d4df4937462f991c5bd2a1972747d3ce034862d813e8d4d |
| POLL_INTERVAL_SECONDS | 15 |
| AGENCY_TIMEZONE | America/New_York |
| Source | MBTA GTFS-Realtime telemetry |
| Sample | N1 |
| File | telemetry_sample_N1.parquet |
| Analysis timezone | America/New_York |
| Polling interval used | 15 seconds |
| Time range | 1787095182299 to 1787095810407 |
| Rows | 24,000 |